Data Preperation


In [1]:
# Install missing packages into the active notebook kernel (safe to re-run)
import importlib
import subprocess
import sys

REQUIRED = ["seaborn"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required plotting packages are already installed.")

All required plotting packages are already installed.


In [2]:
# Standard library
import glob
import warnings
warnings.filterwarnings('ignore')

# Numerics / data
import numpy as np
import pandas as pd

# ── Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Global settings
SEED      = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size'      : 11,
})

print('All imports successful.')
print(f'pandas  {pd.__version__}  |  numpy  {np.__version__}')

All imports successful.
pandas  2.3.2  |  numpy  2.2.3


Loading datasets


In [3]:
import io
import os
import zipfile
from pathlib import Path

# ── Robust file paths (Colab + local) ─────────────────────────────────────────
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def resolve_zip_path(filename: str) -> str:
    """Return first existing path for a dataset zip across common environments."""
    cwd = Path.cwd()
    candidates = [
        cwd / filename,
        cwd.parent / filename,
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
    ]

    if running_in_colab():
        candidates.extend(
            [
                Path("/content") / filename,
                Path("/content") / "drive" / "MyDrive" / filename,
                Path("/content") / "drive" / "MyDrive" / "DS3000" / filename,
            ]
        )

    for path in candidates:
        if path.exists():
            return str(path)

    checked = "\n".join(f"- {p}" for p in candidates)
    raise FileNotFoundError(
        f"Could not locate {filename}. Checked:\n{checked}\n"
        "If needed, place ZIPs in repo root, data/, or /content/drive/MyDrive in Colab."
    )


DEMO_ZIP = resolve_zip_path("DemoStats_2025_CSV_fixed.zip")
SPEND_ZIP = resolve_zip_path("HouseholdSpend_2025_CSV_fixed.zip")

DOC_KEYWORDS = ("metadata", "variables list", "variables change list", "release notes")


def should_skip_csv(path_in_zip: str) -> tuple[bool, str]:
    lower = path_in_zip.lower()
    base = os.path.basename(lower)

    if "documentation/" in lower:
        return True, "inside Documentation/"
    if not lower.endswith(".csv"):
        return True, "not a CSV"
    if any(k in base for k in DOC_KEYWORDS):
        return True, "documentation/metadata CSV"
    return False, ""


def is_data_csv(path_in_zip: str) -> bool:
    """Temporary safeguard: include only data GEO tables."""
    lower = path_in_zip.lower()
    base = os.path.basename(lower)
    return base.endswith("_geo.csv") or "_geo_p" in base


def read_csv_with_geo_filter(file_obj, encoding: str, geo_value: str = "PRCDDA") -> pd.DataFrame:
    """Chunked reader to reduce memory; keeps only PRCDDA rows when GEO exists."""
    chunks = []
    for chunk in pd.read_csv(file_obj, encoding=encoding, low_memory=False, chunksize=100000):
        if "GEO" in chunk.columns:
            chunk = chunk.loc[chunk["GEO"] == geo_value]
        chunks.append(chunk)

    if not chunks:
        return pd.DataFrame()
    return pd.concat(chunks, ignore_index=True)


def load_dataset(zip_path: str, label: str, encoding: str = "latin-1") -> pd.DataFrame:
    """Load only valid data tables from ZIP, with explicit include/skip/failure logs."""
    print(f"  Opening: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

    included_frames = []
    included_names = []
    skipped = []
    failures = []

    with zipfile.ZipFile(zip_path, "r") as zf:
        entries = zf.namelist()
        print(f"  Entries in archive ({len(entries)} total):")
        for e in entries:
            print(f"    {e}")

        for entry in zf.infolist():
            name = entry.filename
            skip, reason = should_skip_csv(name)
            if skip:
                if name.lower().endswith(".csv"):
                    skipped.append((name, reason))
                continue

            if not is_data_csv(name):
                skipped.append((name, "not a GEO data table"))
                continue

            try:
                with zf.open(entry) as f:
                    df = read_csv_with_geo_filter(f, encoding=encoding, geo_value="PRCDDA")
                included_frames.append(df)
                included_names.append(name)
                print(f"    INCLUDE ✓ {os.path.basename(name):50s} {df.shape}")
            except Exception as exc:
                failures.append((name, str(exc)))
                print(f"    FAIL ✗ {os.path.basename(name):50s} {exc}")

    print("\n  Included files:")
    for n in included_names:
        print(f"    - {n}")

    print("\n  Skipped CSV files:")
    if skipped:
        for n, r in skipped:
            print(f"    - {n}  [{r}]")
    else:
        print("    - none")

    print("\n  Failed files:")
    if failures:
        for n, e in failures:
            print(f"    - {n}\n      error: {e}")
    else:
        print("    - none")

    if not included_frames:
        raise ValueError(f"No readable data CSV files included for {label}.")

    # Keep a conservative combine strategy to avoid huge memory growth.
    # If two parts (e.g., P1/P2) share CODE+GEO, merge wide; otherwise stack rows.
    if len(included_frames) == 2 and all(
        col in included_frames[0].columns and col in included_frames[1].columns
        for col in ("CODE", "GEO")
    ):
        left = included_frames[0]
        right = included_frames[1]
        overlap = [c for c in right.columns if c in left.columns and c not in ("CODE", "GEO")]
        if overlap:
            right = right.drop(columns=overlap)
        combined = left.merge(right, on=["CODE", "GEO"], how="inner")
    else:
        combined = pd.concat(included_frames, ignore_index=True, sort=False)

    print(f"\n  → Combined shape for {label}: {combined.shape}")
    return combined


print("=" * 60)
print("Loading DemoStats …")
print("=" * 60)
demo_raw = load_dataset(DEMO_ZIP, label="DemoStats")

print("\n" + "=" * 60)
print("Loading HouseholdSpend …")
print("=" * 60)
spend_raw = load_dataset(SPEND_ZIP, label="HouseholdSpend")

Loading DemoStats …
  Opening: c:\Users\Subhr\datasci3000-housingburden\DemoStats_2025_CSV_fixed.zip  (504.5 MB)
  Entries in archive (12 total):
    DemoStats_(DST)_2020_2025_GEO.csv
    DemoStats_(DST)_2025_2025_GEO.csv
    DemoStats_(DST)_2028_2025_GEO.csv
    DemoStats_(DST)_2030_2025_GEO.csv
    DemoStats_(DST)_2035_2025_GEO.csv
    DemoStats_2025_TRENDS_GEO.csv
    Documentation/
    Documentation/Canada End-User Data product License Agreement 2025.pdf
    Documentation/DemoStats 2025 - Metadata.csv
    Documentation/DemoStats 2025 - Release Notes.pdf
    Documentation/DemoStats 2025 - Technical Document.pdf
    Documentation/DemoStats 2025 - Variables List.csv
    ✓ DemoStats_(DST)_2020_2025_GEO.csv                   (77263, 468)
    ✗ Could not read DemoStats_(DST)_2025_2025_GEO.csv: Unable to allocate 4.98 GiB for an array with shape (704, 950157) and data type int64
    ✓ DemoStats_(DST)_2028_2025_GEO.csv                   (77263, 468)
    ✓ DemoStats_(DST)_2030_2025_GEO.csv 

KeyboardInterrupt: 